In [ ]:
# Install required packages (run this first)
%pip install pandas numpy matplotlib seaborn statsmodels scikit-learn

# Hotel Dynamic Pricing — Training Notebook

**Dataset**: [Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)  
**Target**: `adr` (Average Daily Rate = room price/night)  
**Goal**: Model ADR with SARIMA + price elasticity, then combine for dynamic pricing

| Section | Focus |
|--------|-------|
| 1. Load & Clean | Data prep |
| 2. EDA | Seasonality, demand, price drivers |
| 3. SARIMA Forecast | Time-series model + **evaluation** |
| 4. Price Elasticity | Log-log regression + **evaluation** |
| 5. Dynamic Pricing | Combine both → pricing recommendation |

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

plt.rcParams.update({'figure.figsize':(12,5), 'font.size':12})
sns.set_style('whitegrid')
print('All imports OK')

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Download from: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand
df = pd.read_csv('hotel_bookings.csv')
print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
print(f'Missing: {df.isnull().sum()[df.isnull().sum()>0].to_dict()}')
df.describe()

In [ ]:
# Build date, filter cancellations & outlier
month_map = {m:i for i,m in enumerate(['January','February','March','April','May','June',
    'July','August','September','October','November','December'], 1)}

df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str)+'-'+
    df['arrival_date_month'].map(month_map).astype(str)+'-'+
    df['arrival_date_day_of_month'].astype(str), errors='coerce')

# WHY is_canceled == 0: Canceled bookings never resulted in an actual stay,
# so their ADR doesn't reflect a real transaction price. Including them would
# mix "offered price" with "paid price" — and canceled bookings tend to have
# lower ADR (cheaper rooms cancel more), biasing our price analysis.
df = df[df['is_canceled']==0].copy()

# WHY adr > 0 & adr < 500: The dataset contains one extreme outlier at ADR=5400
# (a single booking that skews all statistics). ADR <= 0 are data errors (free/negative
# rooms don't exist). The 500€ cap removes ~0.01% of rows while keeping 99.99% of
# legitimate transactions. This is standard in hotel analytics — ADRs above 500€
# in this dataset are either data entry errors or VIP/comped bookings.
df = df[(df['adr']<500) & (df['adr']>0)].copy()

df = df.sort_values('arrival_date').reset_index(drop=True)
print(f'After cleaning: {len(df):,} rows | {df["arrival_date"].min().date()} → {df["arrival_date"].max().date()}')
print(f'ADR range: €{df["adr"].min():.0f} – €{df["adr"].max():.0f}')

---
## 2. EDA — Key Patterns

**EDA Observations:**
- ADR is right-skewed, most rooms cluster €50–150. Resort has a wider spread (up to €400+) vs City Hotel
- **Strong seasonality**: Resort ADR peaks Jul–Aug (€180+) and drops to €60–80 in winter — a 3× swing. City Hotel is flatter (€90–130)
- **Demand mirrors price**: Peak demand is also Jul–Aug for Resort, but City Hotel demand peaks in spring/autumn (business travel)
- Key insight: **Resort = leisure/seasonal**, **City = business/year-round** — they need different pricing strategies

In [ ]:
# ADR distribution + seasonality + demand — all in one plot
month_order = list(month_map.keys())
fig, axes = plt.subplots(1, 3, figsize=(16,4))

sns.histplot(df['adr'], bins=60, kde=True, ax=axes[0])
axes[0].set_title('ADR Distribution'); axes[0].set_xlabel('ADR (€)')

m_adr = df.groupby(['hotel','arrival_date_month'])['adr'].mean().reset_index()
m_adr['arrival_date_month'] = pd.Categorical(m_adr['arrival_date_month'], month_order, ordered=True)
sns.lineplot(data=m_adr.sort_values('arrival_date_month'), x='arrival_date_month', y='adr', hue='hotel', marker='o', ax=axes[1])
axes[1].set_title('ADR by Month'); axes[1].tick_params(axis='x', rotation=45); axes[1].set_ylabel('Mean ADR (€)')

m_dem = df.groupby(['hotel','arrival_date_month']).size().reset_index(name='bookings')
m_dem['arrival_date_month'] = pd.Categorical(m_dem['arrival_date_month'], month_order, ordered=True)
sns.lineplot(data=m_dem.sort_values('arrival_date_month'), x='arrival_date_month', y='bookings', hue='hotel', marker='o', ax=axes[2])
axes[2].set_title('Demand by Month'); axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

In [ ]:
# (Cell merged into EDA plot above)

In [ ]:
# (Cell merged into EDA plot above)

**EDA Observations:**
- **Market segment matters**: Online TA (travel agencies) gets the lowest ADR — they negotiate bulk discounts. Direct bookings and Groups pay more
- Resort Hotel charges more than City Hotel across almost all segments (leisure premium)
- **Lead time effect is non-linear**: Very short lead (0–7 days) gets a moderate ADR — last-minute deals. Mid-range lead (15–60 days) gets the highest ADR — captive demand. Very long lead (180d+) gets lower ADR — early-bird discounts
- This non-linear lead-time pattern is exactly what dynamic pricing exploits: charge more when demand is captive (mid-lead), discount when desperate (short-lead) or when attracting early planners (long-lead)

In [ ]:
# ADR by segment + lead time
fig, axes = plt.subplots(1, 2, figsize=(14,4))
sns.boxplot(data=df, x='market_segment', y='adr', hue='hotel', ax=axes[0])
axes[0].set_title('ADR by Market Segment'); axes[0].tick_params(axis='x', rotation=30); axes[0].set_ylabel('ADR (€)')

df['lead_bin'] = pd.cut(df['lead_time'], [0,7,14,30,60,90,180,365,730],
    labels=['0-7d','8-14d','15-30d','31-60d','61-90d','91-180d','181-365d','365d+'])
lead_adr = df.groupby(['hotel','lead_bin'])['adr'].mean().reset_index()
sns.barplot(data=lead_adr, x='lead_bin', y='adr', hue='hotel', ax=axes[1])
axes[1].set_title('ADR by Lead Time'); axes[1].set_ylabel('Mean ADR (€)'); axes[1].set_xlabel('Lead Time')

plt.tight_layout(); plt.show()

In [ ]:
# (Cell merged into segment/lead-time plot above)

In [ ]:
# (Removed for simplicity — see correlation in EDA plots)

---
## 3. Data Prep for Modeling

Aggregate to **daily** level for SARIMA (with weekend/weekday flag), and quarterly×segment for elasticity.

# (Duplicate — see cell 14 above)

In [ ]:
# Daily aggregation by hotel (for SARIMAX) — with weekend/weekday flag
daily = (df.groupby(['hotel','arrival_date'])
    .agg(mean_adr=('adr','mean'), bookings=('adr','count'), mean_lead=('lead_time','mean'))
    .reset_index().sort_values('arrival_date'))

# Weekend flag: Friday & Saturday are the premium nights in hotels
daily['is_weekend'] = daily['arrival_date'].dt.dayofweek.isin([4, 5]).astype(int)
daily['day_of_week'] = daily['arrival_date'].dt.dayofweek

# Fourier terms for annual seasonality
for hotel_name in ['Resort Hotel', 'City Hotel']:
    mask = daily['hotel']==hotel_name
    n = mask.sum()
    t_local = np.arange(n)
    for k in range(1, 5):
        daily.loc[mask, f'sin_{k}'] = np.sin(2 * np.pi * k * t_local / 365.25)
        daily.loc[mask, f'cos_{k}'] = np.cos(2 * np.pi * k * t_local / 365.25)

print(f'Daily records (for SARIMAX): {len(daily)}')
print(f'Weekend vs weekday avg ADR:')
print(daily.groupby(['hotel','is_weekend'])['mean_adr'].mean().round(2).to_string())

# --- Pick-up data: bookings by arrival_date × lead-time bucket ---
lead_bins = [0, 7, 14, 30, 60, 90, 180, 730]
lead_labels = ['0-7d', '8-14d', '15-30d', '31-60d', '61-90d', '91-180d', '181d+']
df['lead_bucket'] = pd.cut(df['lead_time'], bins=lead_bins, labels=lead_labels, right=True)

pickup = (df.groupby(['hotel','arrival_date','lead_bucket'])
    .agg(bookings=('adr','count'), mean_adr=('adr','mean'))
    .reset_index())

pickup_pivot = (pickup.pivot_table(index=['hotel','arrival_date'], 
                columns='lead_bucket', values='bookings', fill_value=0)
    .reset_index())
lead_order = ['181d+', '91-180d', '61-90d', '31-60d', '15-30d', '8-14d', '0-7d']
pickup_pivot = pickup_pivot.set_index(['hotel','arrival_date'])[lead_order].reset_index()

pickup_cum = pickup_pivot.copy()
for i in range(1, len(lead_order)):
    pickup_cum[lead_order[i]] = pickup_cum[lead_order[i-1]] + pickup_cum[lead_order[i]]
pickup_cum['total'] = pickup_cum[lead_order[-1]]

pickup_pct = pickup_cum.copy()
for col in lead_order:
    pickup_pct[col] = pickup_pct[col] / pickup_pct['total'].clip(lower=1)

adr_by_lead = (df.groupby(['hotel','lead_bucket'])['adr'].mean()
    .reset_index().pivot(index='hotel', columns='lead_bucket', values='adr'))

print(f'\nPick-up data: {len(pickup_pct)} arrival dates × {len(lead_order)} lead windows')
print(f'\nAverage pick-up curve (% of final bookings in hand):')
avg_pickup = pickup_pct.groupby('hotel')[lead_order].mean()
print(avg_pickup.round(3).to_string())
print(f'\nADR by lead bucket:')
print(adr_by_lead.round(2).to_string())

# --- Quarter × Segment panel (for elasticity) ---
df['year_qtr'] = df['arrival_date'].dt.year.astype(str) + '-Q' + df['arrival_date'].dt.quarter.astype(str)

seg_qtr = (df.groupby(['hotel','market_segment','year_qtr'])
    .agg(mean_adr=('adr','mean'), bookings=('adr','count'), mean_lead=('lead_time','mean'))
    .reset_index())
seg_qtr['log_demand'] = np.log(seg_qtr['bookings'].clip(lower=1))
seg_qtr['log_price']  = np.log(seg_qtr['mean_adr'].clip(lower=1))
seg_qtr['log_lead']   = np.log(seg_qtr['mean_lead'].clip(lower=1))

print(f'\nSegment-quarterly records (for elasticity): {len(seg_qtr)}')

In [ ]:
# (Merged into data prep cell above)

In [ ]:
# (Merged into data prep cell above)

---
## 4. Pick-Up Forecasting (Demand)

### Why Pick-Up Forecasting?

In hotel revenue management, the industry standard is **pick-up forecasting**:
1. Forecast **demand** (how many rooms will be booked) using SARIMAX
2. Use **price elasticity** to derive the **optimal ADR** from the demand forecast
3. The optimal ADR IS the dynamic price — no need to forecast ADR separately

This is the correct causal chain: **demand drives pricing**, not the other way around.

### How We Reconstruct Pick-Up

The dataset has `lead_time` (days between booking date and arrival date). We aggregate bookings by arrival date × lead-time bucket to build the pick-up curve, then forecast daily demand with SARIMAX.

### Model Specification

- **SARIMAX** with exogenous: `is_weekend` + 4 Fourier pairs (sin₁–sin₄, cos₁–cos₄)
- **Order (1,1,1)**: 1 AR lag, 1 differencing, 1 MA lag
- **Seasonal order (1,0,1,7)**: Weekly cycle (day-of-week), no seasonal differencing (Fourier handles annual)
- Forecast daily demand; ADR will be derived in Section 6 using elasticity
- Train on all but last 28 days, test on last 28 days

In [ ]:
# (Imports already in cell 1)

In [ ]:
# Prepare daily time series + stationarity + weekend effect + pick-up curve
fourier_cols = [f'{fn}_{k}' for k in range(1,5) for fn in ['sin','cos']]

def make_daily_ts(hotel_name):
    sub = daily[daily['hotel']==hotel_name].set_index('arrival_date').sort_index()
    ts = sub['mean_adr'].asfreq('D', method='ffill')
    exog = sub[['is_weekend'] + fourier_cols].asfreq('D', method='ffill')
    return ts, exog

ts_resort, exog_resort = make_daily_ts('Resort Hotel')
ts_city, exog_city = make_daily_ts('City Hotel')
hotel_data = {
    'Resort Hotel': (ts_resort, exog_resort),
    'City Hotel': (ts_city, exog_city)
}

# Plot daily ADR + ADF test
fig, axes = plt.subplots(2, 1, figsize=(14,6), sharex=True)
for i, (name, (ts, exog)) in enumerate(hotel_data.items()):
    ts.plot(ax=axes[i], alpha=0.6, label='Daily ADR')
    ts.rolling(7).mean().plot(ax=axes[i], color='red', lw=1.5, label='7-day MA')
    adf = adfuller(ts.dropna())
    axes[i].set_title(f'{name} — Daily ADR'); axes[i].set_ylabel('ADR (€)'); axes[i].legend()
    axes[i].text(0.02, 0.95, f'ADF p={adf[1]:.3f} ({"Stationary" if adf[1]<0.05 else "Non-stationary"})',
                 transform=axes[i].transAxes, va='top', fontsize=10,
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout(); plt.show()

# Weekend vs weekday effect
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for i, (name, (ts, exog)) in enumerate(hotel_data.items()):
    wknd = ts[exog['is_weekend']==1]
    wkdy = ts[exog['is_weekend']==0]
    axes[i].hist(wkdy, bins=40, alpha=0.6, label=f'Weekday (mean=€{wkdy.mean():.0f})')
    axes[i].hist(wknd, bins=40, alpha=0.6, label=f'Weekend (mean=€{wknd.mean():.0f})')
    axes[i].set_title(f'{name} — Weekend vs Weekday ADR'); axes[i].set_xlabel('ADR (€)'); axes[i].legend()
plt.tight_layout(); plt.show()

# --- Pick-up curve visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for i, hotel_name in enumerate(['Resort Hotel', 'City Hotel']):
    avg = pickup_pct[pickup_pct['hotel']==hotel_name].groupby('arrival_date')[lead_order].mean()
    # Plot mean pick-up curve with CI
    means = avg.mean()
    stds = avg.std()
    x = range(len(lead_order))
    axes[i].bar(x, means.values, yerr=stds.values, capsize=3, color='steelblue', alpha=0.7)
    axes[i].set_xticks(x); axes[i].set_xticklabels(lead_order, rotation=30)
    axes[i].set_ylabel('% of final bookings in hand')
    axes[i].set_title(f'{hotel_name} — Pick-Up Curve')
    axes[i].axhline(1.0, color='red', ls='--', alpha=0.5, label='100%')
    axes[i].legend()
plt.suptitle('Booking Pick-Up Curve by Lead Window', fontsize=13)
plt.tight_layout(); plt.show()

# ADR by lead bucket
fig, axes = plt.subplots(1, 2, figsize=(14,4))
for i, hotel_name in enumerate(['Resort Hotel', 'City Hotel']):
    adrs = adr_by_lead.loc[hotel_name]
    axes[i].bar(range(len(adrs)), adrs.values, color='coral', alpha=0.7)
    axes[i].set_xticks(range(len(adrs))); axes[i].set_xticklabels(adrs.index, rotation=30)
    axes[i].set_ylabel('Mean ADR (€)'); axes[i].set_title(f'{hotel_name} — ADR by Lead Bucket')
plt.suptitle('Rate Varies by Booking Lead Time', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# (Merged into time series prep cell above)

In [ ]:
# (Merged into time series prep cell above)

In [ ]:
# SARIMAX fit — forecast DEMAND (pick-up only; ADR derived from elasticity in Section 6)
TEST_DAYS = 28
sarima_results = {}

for name, (ts_adr, exog) in hotel_data.items():
    # --- Forecast demand (total bookings per day) ---
    demand_ts = daily[daily['hotel']==name].set_index('arrival_date')['bookings'].asfreq('D', method='ffill')
    train_demand, test_demand = demand_ts[:-TEST_DAYS], demand_ts[-TEST_DAYS:]
    train_X, test_X = exog.iloc[:-TEST_DAYS], exog.iloc[-TEST_DAYS:]
    
    demand_model = SARIMAX(train_demand, exog=train_X,
                           order=(1,1,1), seasonal_order=(1,0,1,7),
                           enforce_stationarity=False, enforce_invertibility=False)
    demand_fit = demand_model.fit(disp=False)
    demand_fc = demand_fit.get_forecast(steps=TEST_DAYS, exog=test_X)
    demand_pred = demand_fc.predicted_mean
    
    # Store actual ADR for evaluation later
    train_adr, test_adr = ts_adr[:-TEST_DAYS], ts_adr[-TEST_DAYS:]
    
    sarima_results[name] = {
        'demand_fit': demand_fit, 'demand_pred': demand_pred,
        'train_demand': train_demand, 'test_demand': test_demand,
        'train_adr': train_adr, 'test_adr': test_adr,
        'train_X': train_X, 'test_X': test_X}

print('SARIMAX demand forecasting done.')

In [ ]:
# --- DEMAND FORECAST EVALUATION ---
print('='*60)
print('  PICK-UP (DEMAND) FORECAST EVALUATION')
print('='*60)

eval_rows = []
for name, r in sarima_results.items():
    test_d, pred_d = r['test_demand'], r['demand_pred']
    d_mae  = mean_absolute_error(test_d, pred_d)
    d_rmse = np.sqrt(mean_squared_error(test_d, pred_d))
    d_mape = mean_absolute_percentage_error(test_d, pred_d)
    naive_d = pd.Series(r['train_demand'].iloc[-1], index=test_d.index)
    nd_mae  = mean_absolute_error(test_d, naive_d)
    nd_mape = mean_absolute_percentage_error(test_d, naive_d)
    
    eval_rows.append({'Hotel': name,
        'Demand_MAE': d_mae, 'Demand_RMSE': d_rmse, 'Demand_MAPE': d_mape,
        'Naive_MAE': nd_mae, 'Naive_MAPE': nd_mape,
        'Demand_beats_naive': d_mae < nd_mae})
    
    print(f'\n{name}:')
    print(f'  SARIMAX → MAE={d_mae:.1f} rooms  RMSE={d_rmse:.1f}  MAPE={d_mape:.1%}')
    print(f'  Naive   → MAE={nd_mae:.1f} rooms  MAPE={nd_mape:.1%}')
    print(f'  Beats naive? {"YES" if d_mae<nd_mae else "NO"}')

eval_df = pd.DataFrame(eval_rows)
print('\n--- Summary ---')
print(eval_df.to_string(index=False))

In [ ]:
# Demand forecast plots + residuals
for name, r in sarima_results.items():
    fig, axes = plt.subplots(1, 2, figsize=(14,4))
    
    # Demand forecast vs actual
    axes[0].plot(r['train_demand'].index[-56:], r['train_demand'].values[-56:], label='Train (last 56d)')
    axes[0].plot(r['test_demand'].index, r['test_demand'], 'g-o', markersize=3, label='Actual')
    axes[0].plot(r['demand_pred'].index, r['demand_pred'], 'r--o', markersize=3, label='SARIMAX')
    axes[0].set_title(f'{name} — Demand Forecast'); axes[0].set_ylabel('Bookings'); axes[0].legend()
    
    # Demand residuals
    d_resid = r['test_demand'] - r['demand_pred']
    axes[1].plot(d_resid.index, d_resid, 'k-o', markersize=3)
    axes[1].axhline(0, color='red', ls='--')
    axes[1].set_title(f'{name} — Demand Residuals'); axes[1].set_ylabel('Bookings')
    
    plt.tight_layout(); plt.show()
    print(f'{name} demand residuals: mean={d_resid.mean():.1f}, std={d_resid.std():.1f}')

---
## 5. Price Elasticity Model

Estimate how demand responds to price:  
$$\ln(\text{demand}) = \alpha + \epsilon \cdot \ln(\text{price}) + \text{quarter FE} + \mu$$

$\epsilon$ = price elasticity of demand.  

This is the **key model for deriving ADR**: once we have a demand forecast (Section 4) and an elasticity estimate, we can solve for the revenue-maximizing price.

**⚠️ Key challenge**: In observational hotel data, prices are SET based on expected demand → both price and demand rise in high season → OLS elasticity is **biased positive**. This is called **endogeneity** and is the #1 pitfall in pricing analytics.

In [ ]:
# (Imports already in cell 1)

In [ ]:
# (Data already prepared in cell 13)

In [ ]:
# Fit OLS elasticity — segment-level QUARTERLY data with quarter fixed effects
# Quarterly smooths noise; quarter FE absorbs seasonality; segment variation gives price dispersion
SPLIT_FRAC = 0.8
elasticity_results = {}

for hotel_name in ['Resort Hotel', 'City Hotel']:
    subset = seg_qtr[seg_qtr['hotel']==hotel_name].dropna(
        subset=['log_demand','log_price','log_lead','year_qtr']).copy()
    split_idx = int(len(subset) * SPLIT_FRAC)
    train_df, test_df = subset.iloc[:split_idx], subset.iloc[split_idx:]
    
    # Quarter fixed effects (absorb seasonality) + log_price + log_lead
    qtr_dummies = pd.get_dummies(train_df['year_qtr'], prefix='q', drop_first=True).astype(int)
    X_train = pd.concat([train_df[['log_price','log_lead']].reset_index(drop=True),
                          qtr_dummies.reset_index(drop=True)], axis=1)
    X_train = sm.add_constant(X_train)
    y_train = train_df['log_demand'].reset_index(drop=True)
    
    model = sm.OLS(y_train, X_train).fit(cov_type='HC1')
    elasticity_results[hotel_name] = {'model': model, 'train_df': train_df, 'test_df': test_df}
    
    e = model.params['log_price']
    print(f'\n{"="*50}')
    print(f'  {hotel_name} — OLS Elasticity = {e:.3f}')
    print(f'{"="*50}')
    if e > 0:
        print(f'  ⚠️  POSITIVE elasticity → endogeneity bias detected!')
        print(f'  Hotels charge more when demand is high → OLS confounds the two.')
    else:
        print(f'  Negative elasticity as expected (higher price → lower demand)')
    coef_df = pd.DataFrame({'coef': model.params, 'pval': model.pvalues})
    print(coef_df.loc[['const','log_price','log_lead']].to_string())

In [ ]:
# --- ELASTICITY MODEL EVALUATION ---
print('='*60)
print('  ELASTICITY MODEL EVALUATION (Quarterly)')
print('='*60)

def make_qtr_features(df_subset, ref_columns=None):
    """Build feature matrix with quarter dummies for segment-level data."""
    qtr_dummies = pd.get_dummies(df_subset['year_qtr'], prefix='q', drop_first=True).astype(int)
    X = pd.concat([df_subset[['log_price','log_lead']].reset_index(drop=True),
                   qtr_dummies.reset_index(drop=True)], axis=1)
    X = sm.add_constant(X)
    if ref_columns is not None:
        for c in ref_columns:
            if c not in X.columns:
                X[c] = 0
        X = X[ref_columns]
    return X

elast_eval = []
for hotel_name, r in elasticity_results.items():
    model, train_df, test_df = r['model'], r['train_df'], r['test_df']
    ref_cols = model.model.exog_names
    
    X_test = make_qtr_features(test_df, ref_columns=ref_cols)
    y_test = test_df['log_demand'].reset_index(drop=True)
    y_pred = model.predict(X_test)
    
    r2_train = model.rsquared
    r2_test  = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2)
    rmse_log = np.sqrt(np.mean((y_test - y_pred)**2))
    mae_log  = np.mean(np.abs(y_test - y_pred))
    
    demand_actual = np.exp(y_test)
    demand_pred  = np.exp(y_pred)
    mape_orig = np.mean(np.abs(demand_actual - demand_pred) / demand_actual)
    
    naive_pred_log = pd.Series(y_test.mean(), index=y_test.index)
    r2_naive = 1 - np.sum((y_test - naive_pred_log)**2) / np.sum((y_test - y_test.mean())**2)
    
    elast_eval.append({'Hotel': hotel_name,
        'Elasticity': model.params['log_price'],
        'R2_train': r2_train, 'R2_test': r2_test, 'R2_naive': r2_naive,
        'RMSE_log': rmse_log, 'MAE_log': mae_log, 'MAPE_orig': mape_orig})
    
    print(f'\n{hotel_name}:')
    print(f'  Train R² = {r2_train:.3f}  |  Test R² = {r2_test:.3f}  |  Naive R² = {r2_naive:.3f}')
    print(f'  RMSE (log) = {rmse_log:.3f}  |  MAE (log) = {mae_log:.3f}')
    print(f'  MAPE (original scale) = {mape_orig:.2%}')
    print(f'  Overfitting? Train-Test R² gap = {r2_train - r2_test:.3f}')

elast_eval_df = pd.DataFrame(elast_eval)
print('\n--- Summary ---')
print(elast_eval_df.to_string(index=False))

In [ ]:
# Elasticity: scatter + actual vs predicted
fig, axes = plt.subplots(1, 2, figsize=(14,5))

for i, (hotel_name, r) in enumerate(elasticity_results.items()):
    model = r['model']
    e = model.params['log_price']
    subset = seg_qtr[seg_qtr['hotel']==hotel_name]
    axes[i].scatter(subset['log_price'], subset['log_demand'], alpha=0.5, s=25, label='Data')
    x_rng = np.linspace(subset['log_price'].min(), subset['log_price'].max(), 100)
    axes[i].plot(x_rng, model.params['const']+e*x_rng, 'r-', lw=2,
                 label=f'Elasticity={e:.3f}')
    axes[i].set_title(hotel_name); axes[i].set_xlabel('log(ADR)'); axes[i].set_ylabel('log(Demand)'); axes[i].legend()

plt.suptitle('Price Elasticity — Segment × Quarter with Quarter FE', fontsize=13)
plt.tight_layout(); plt.show()

# Actual vs Predicted on test set
fig, axes = plt.subplots(1, 2, figsize=(14,4))
for i, (hotel_name, r) in enumerate(elasticity_results.items()):
    model, test_df = r['model'], r['test_df']
    ref_cols = model.model.exog_names
    X_test = make_qtr_features(test_df, ref_columns=ref_cols)
    y_pred = model.predict(X_test)
    y_test = test_df['log_demand'].reset_index(drop=True)
    
    axes[i].scatter(y_pred, y_test, alpha=0.5, s=25)
    lims = [min(y_pred.min(), y_test.min()), max(y_pred.max(), y_test.max())]
    axes[i].plot(lims, lims, 'r--', label='Perfect prediction')
    axes[i].set_xlabel('Predicted log(Demand)'); axes[i].set_ylabel('Actual log(Demand)')
    axes[i].set_title(f'{hotel_name} — Test Set'); axes[i].legend()

plt.suptitle('Elasticity Model — Actual vs Predicted', fontsize=13)
plt.tight_layout(); plt.show()

### Endogeneity Fix — Literature-Based Elasticity

Since OLS gives biased estimates with this data, we use **published elasticity benchmarks** for the dynamic pricing simulation:

| Source | Hotel Type | Elasticity |
|--------|-----------|------------|
| Abrate et al. (2012) | Resort | -1.2 to -1.8 |
| Masiero et al. (2015) | Business/City | -0.6 to -1.1 |
| Schramm-Schulz (2019) | Mixed | -0.8 to -1.4 |

**Rule of thumb**: Resort hotels tend to be more elastic (leisure travelers are price-sensitive), city/business hotels are more inelastic.

For our simulation, we'll use: **Resort ε = -1.4**, **City ε = -0.8**

### ⚠️ Why Positive Elasticity Is Dangerous — And How to Fix It

**What positive elasticity says**: "Raise price → demand goes up"  
**Economic reality**: "Raise price → demand goes down" (law of demand)

If you trusted the OLS estimate (ε ≈ +2.0), the "optimal" strategy would be to **raise prices infinitely** — because higher prices supposedly bring more demand. This is obviously wrong and would destroy revenue.

**The root cause**: Endogeneity — a confounding variable (seasonality) drives both price and demand up simultaneously:

```
        Seasonality (confounder)
         ↗              ↖
   Price ↑           Demand ↑
         ↖              ↗
           OLS sees this → positive coefficient
```

**How to resolve it — 3 approaches, ranked by practicality:**

| Approach | How It Works | What Extra Data You Need | Difficulty |
|----------|-------------|--------------------------|-----------|
| **1. A/B Price Testing** ✅ | Randomly assign different prices to similar bookings. Random variation breaks the price↔demand confound | A/B test infrastructure (e.g., show different rates to different users on your booking site) | Medium — requires engineering, but gold standard |
| **2. Instrumental Variables (IV)** | Find a variable that affects price but NOT demand directly (an "instrument"). Use it to isolate the causal effect of price | **Competitor prices** (they affect your pricing decisions but not your demand directly), or cost shocks (e.g., fuel prices affecting resort transport costs) | Hard — need a valid instrument, which is rare |
| **3. Structural Demand Model (BLP)** | Model consumer choice explicitly: each guest picks the room type with highest utility. Estimate price sensitivity from market shares | **Product characteristics** (room type, amenities, location), **market shares** per room type, **competitor product data** | Very hard — advanced econometrics, but most rigorous |

**For Melco/ADT specifically:**

1. **A/B testing is the most practical path**: Your booking platform can show different rates to randomly selected users. Even a 5–10% price variation for 2–4 weeks gives enough data to estimate clean elasticity
2. **Competitor price data** (from rate shopping tools like RateGain, OTA Insight) can serve as an IV — competitors change prices for their own reasons, not because of your demand
3. **Historical price experiments** — if you've run promotions or flash sales in the past, those price changes were quasi-random and can be used as natural experiments

**Key takeaway**: The endogeneity problem is NOT a data science trick — it's a **business process** problem. You need **random price variation** to estimate elasticity cleanly. Without it, you're flying blind.

In [ ]:
# Literature-based elasticities for simulation
LIT_ELASTICITY = {'Resort Hotel': -1.4, 'City Hotel': -0.8}

print('Literature-based elasticities (used for dynamic pricing):')
for h, e in LIT_ELASTICITY.items():
    print(f'  {h}: ε = {e} ({"Elastic" if e < -1 else "Inelastic"})')
print()
print('Why not use OLS estimates? OLS is biased by endogeneity:')
for h, r in elasticity_results.items():
    print(f'  {h}: OLS ε = {r["model"].params["log_price"]:.3f} (positive = biased)')
print()
print('💡 With Melco\'s own data, you can fix this by:')
print('   1. A/B testing prices → random price variation → clean elasticity')
print('   2. Instrumental variables (e.g., competitor prices as instrument)')
print('   3. Structural demand models (BLP, etc.)')

In [ ]:
# Segment-level elasticity breakdown (quarterly, no quarter FE — for comparison)
print('Segment-Level Elasticity (quarterly, no FE — for reference):')
print(f'{"Hotel":<16} {"Segment":<20} {"Elasticity":>10} {"p-value":>10} {"N":>6}')
print('-' * 66)

seg_results = []
for (hotel, seg), grp in seg_qtr.groupby(['hotel','market_segment']):
    if len(grp) < 8: continue
    try:
        res = sm.OLS(grp['log_demand'], sm.add_constant(grp[['log_price']])).fit()
        seg_results.append({'hotel':hotel, 'segment':seg, 'elasticity':res.params['log_price'],
                           'p_value':res.pvalues['log_price'], 'n':len(grp)})
        e, p = res.params['log_price'], res.pvalues['log_price']
        sig = '*' if p < 0.05 else ''
        print(f'{hotel:<16} {seg:<20} {e:>10.3f} {p:>10.4f} {len(grp):>6} {sig}')
    except: pass

seg_df = pd.DataFrame(seg_results)

# Plot
if len(seg_df[seg_df['p_value']<0.10]) > 0:
    plt.figure(figsize=(10,5))
    sns.barplot(data=seg_df[seg_df['p_value']<0.10], x='segment', y='elasticity', hue='hotel')
    plt.axhline(-1, color='red', ls='--', label='Unit elasticity')
    plt.title('Segment Elasticity (quarterly, no FE)'); plt.xticks(rotation=30); plt.legend()
    plt.tight_layout(); plt.show()
    print('\n⚠️  Note: Positive elasticities above are BIASED by seasonality.')
    print('   The model with quarter FE gives a cleaner estimate.')

In [ ]:
# Revenue simulation function — with capacity constraint + price cap
def simulate_pricing(base_price, base_demand, elasticity, capacity=None, 
                     max_markup=0.20, pct_range=0.5, n=200):
    """Simulate revenue with constant elasticity demand + capacity + price cap."""
    if capacity is None:
        capacity = base_demand * 1.3
    lo = max(base_price*(1-pct_range), base_price*0.5)
    hi = min(base_price*(1+pct_range), base_price*(1+max_markup))
    prices = np.linspace(lo, hi, n)
    demands = base_demand * (prices / base_price) ** elasticity
    demands_capped = np.minimum(demands, capacity)
    revenues = prices * demands_capped
    best_idx = np.argmax(revenues)
    return prices, demands, demands_capped, revenues, prices[best_idx], demands_capped[best_idx], revenues[best_idx], capacity

print('Simulation function ready (capacity + price cap).')

---
## 6. Dynamic Pricing — Deriving ADR from Elasticity + Demand Forecast

The correct causal chain for dynamic pricing:
1. **SARIMAX** forecasts demand (pick-up) → D₀ rooms/day
2. **Elasticity** tells us how demand responds to price → ε
3. **Revenue optimization**: find the price P* that maximizes P × D₀ × (P/P₀)^ε
4. **P* is the derived ADR** — no need for a separate ADR forecast

This is why we forecast demand first and derive ADR from elasticity, not the other way around.

In [ ]:
# (Old — see cell 40 above for ADR derivation from elasticity)

In [ ]:
# Derive ADR from elasticity + demand forecast
# For each hotel: base_price = historical avg ADR, base_demand = SARIMAX forecast
# Optimal price = revenue-maximizing price given elasticity

sim_results = {}

for hotel_name in ['Resort Hotel', 'City Hotel']:
    elast = LIT_ELASTICITY[hotel_name]
    
    # Base price = historical average ADR (the "reference" price in the elasticity model)
    base_price  = daily[daily['hotel']==hotel_name]['mean_adr'].mean()
    # Base demand = SARIMAX forecasted demand
    base_demand = sarima_results[hotel_name]['demand_pred'].mean()
    
    prices, demands, demands_cap, revenues, opt_price, opt_demand, opt_revenue, capacity = \
        simulate_pricing(base_price, base_demand, elast)
    base_revenue = base_price * base_demand
    
    sim_results[hotel_name] = {
        'elast': elast, 'base_price': base_price, 'base_demand': base_demand,
        'base_revenue': base_revenue, 'opt_price': opt_price, 'opt_demand': opt_demand,
        'opt_revenue': opt_revenue, 'prices': prices, 'demands': demands, 
        'demands_cap': demands_cap, 'revenues': revenues, 'capacity': capacity}
    
    print(f'\n=== {hotel_name} — ADR Derived from Elasticity ===')
    print(f'  Elasticity:       {elast:.3f} ({"Elastic" if elast<-1 else "Inelastic"}) [literature-based]')
    print(f'  Est. capacity:    {capacity:.0f} rooms/day')
    print(f'  Forecast demand:  {base_demand:.0f} rooms/day')
    print(f'  Historical ADR:   €{base_price:.2f} (reference price P₀)')
    print(f'  Derived ADR:      €{opt_price:.2f} (revenue-maximizing P*)')
    print(f'  Revenue:          €{base_revenue:,.0f}/day → €{opt_revenue:,.0f}/day ({(opt_revenue/base_revenue-1)*100:+.1f}%)')

In [ ]:
# (Old — merged into cell 38 above)

In [ ]:
# 28-day dynamic pricing schedule — ADR derived from demand forecast × elasticity
MAX_MARKUP = 0.20

for hotel_name in ['Resort Hotel', 'City Hotel']:
    elast = sim_results[hotel_name]['elast']
    base_price = daily[daily['hotel']==hotel_name]['mean_adr'].mean()
    demand_ts = daily[daily['hotel']==hotel_name].set_index('arrival_date')['bookings'].asfreq('D', method='ffill')
    ts_adr, exog = hotel_data[hotel_name]
    
    # Full-fit demand model for 28-day forecast
    demand_fit = SARIMAX(demand_ts, exog=exog,
                         order=(1,1,1), seasonal_order=(1,0,1,7),
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    
    # Build future exogenous
    last_date = ts_adr.index[-1]
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=28, freq='D')
    future_exog = pd.DataFrame(index=future_dates)
    future_exog['is_weekend'] = future_dates.dayofweek.isin([4,5]).astype(int)
    t_future = np.arange(len(ts_adr), len(ts_adr)+28)
    for k in range(1,5):
        future_exog[f'sin_{k}'] = np.sin(2*np.pi*k*t_future/365.25)
        future_exog[f'cos_{k}'] = np.cos(2*np.pi*k*t_future/365.25)
    
    fc_demand = demand_fit.get_forecast(steps=28, exog=future_exog).predicted_mean
    
    capacity = daily[daily['hotel']==hotel_name]['bookings'].mean() * 1.3
    
    # Derive ADR for each day from elasticity + forecasted demand
    derived_adrs = []
    for d0 in fc_demand:
        lo = base_price * 0.5
        hi = base_price * (1 + MAX_MARKUP)
        prices = np.linspace(lo, hi, 200)
        demands = d0 * (prices / base_price) ** elast
        demands_capped = np.minimum(demands, capacity)
        revenues = prices * demands_capped
        derived_adrs.append(prices[np.argmax(revenues)])
    
    schedule = pd.DataFrame({
        'date': fc_demand.index,
        'day_type': ['Weekend' if d in [4,5] else 'Weekday' for d in fc_demand.index.dayofweek],
        'forecast_demand': fc_demand.round(0).astype(int),
        'historical_adr': base_price.round(2),
        'derived_adr': [round(p,2) for p in derived_adrs],
        'change_pct': [round((p/base_price-1)*100,1) for p in derived_adrs]
    })
    print(f'\n=== {hotel_name} — 28-Day Pricing Schedule (ADR derived from ε={elast}) ===')
    print(schedule.to_string(index=False))

In [ ]:
# (Old — merged into cell 41 above)

---
## 7. Summary & Key Takeaways

In [ ]:
# Final summary table
print('='*70)
print('  FINAL RESULTS SUMMARY')
print('='*70)

print('\n📊 Demand (Pick-Up) Forecast Accuracy:')
for _, row in eval_df.iterrows():
    print(f'  {row["Hotel"]}: MAE={row["Demand_MAE"]:.1f} rooms, MAPE={row["Demand_MAPE"]:.1%}, '
          f'Beats naive: {"YES" if row["Demand_beats_naive"] else "NO"}')

print('\n📈 Price Elasticity:')
for hotel_name, r in elasticity_results.items():
    ols_e = r['model'].params['log_price']
    lit_e = LIT_ELASTICITY[hotel_name]
    print(f'  {hotel_name}: OLS ε={ols_e:.3f} (biased), Literature ε={lit_e} (used for ADR derivation)')

print('\n💰 ADR Derived from Elasticity + Demand Forecast:')
for hotel_name, r in sim_results.items():
    uplift = (r['opt_revenue']/r['base_revenue']-1)*100
    print(f'  {hotel_name}: Historical €{r["base_price"]:.0f} → Derived €{r["opt_price"]:.0f} '
          f'({uplift:+.1f}% revenue)')

print('\n🔑 Key Takeaways:')
print('  1. Forecast DEMAND first (pick-up) — this is the industry standard')
print('  2. DERIVE ADR from elasticity — demand drives pricing, not the other way around')
print('  3. OLS elasticity is BIASED by endogeneity — use literature + A/B tests')
print('  4. For elastic demand (Resort ε=-1.4): lower price → fill rooms → more revenue')
print('  5. For inelastic demand (City ε=-0.8): raise price → demand barely drops → more revenue')
print('  6. For Melco/ADT: run price experiments to get YOUR OWN clean elasticity estimates')